# Epoch Spring Camp 2026 - Take Home Task 3

In this task, you will build and compare two recommender system models:

- **Matrix Factorization (MF)** using a dot product of embeddings  
- **Neural Collaborative Filtering (MLP-based)** using a multi-layer perceptron  

You are provided with:
- Preprocessed interaction data
- Evaluation pipeline

You are expected to implement:
- Negative sampling
- Model architectures
- Training loop

---

The purpose of this task is to understand how neural networks can model user–item interactions beyond simple similarity.

## Imports

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np

## Loading Data

We begin by loading the interaction data.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/interactions - interactions.csv")  # columns: user_id, movie_id

print("Num users:", df['user_id'].nunique())
print("Num items:", df['item_id'].nunique())
print("Num interactions:", len(df))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Num users: 942
Num items: 1447
Num interactions: 55375


Each row represents a **positive interaction** (implicit feedback).

## Train / Test Split

We split the data into training and testing sets.

The model will learn from the training data and be evaluated on unseen interactions in the test set.

In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

## Negative Sampling

The dataset only contains **positive interactions** (user interacted with item).

However, to train a model, we also need **negative examples** (user did not interact with item).

---

### Why do we need this?

We treat recommendation as a **binary classification problem**:

- Positive interaction → label = 1  
- No interaction → label = 0  

Since we don’t have explicit negatives, we **sample them**.

---

### Your Task

For each `(user, item)` interaction:
- Keep it as a **positive sample (label = 1)**
- Sample one or more items the user has **not interacted with**
  - Add them as **negative samples (label = 0)**

---

### Hints

- You can randomly sample items
- Make sure sampled negatives are **not already in the dataset**
- You may use a `set` of `(user, item)` pairs for fast lookup

---

Implement a function:
`sample_negatives(df, num_neg=1)`
that returns a dataframe with columns:
`[user, item, label]`

In [6]:
def sample_negatives(df, num_neg=1):
    # TODO:
    # 1. Create a set of (user, item) interactions
    interactions = set(zip(df['user_id'], df['item_id']))

    data_with_negatives = []
    # 2. For each positive interaction:
    unqiue_items = df['item_id'].unique()
    for user, item in zip(df['user_id'], df['item_id']):
      #    - add (user, item, 1)
      data_with_negatives.append({'user': user, 'item': item, 'label': 1})
      #    - sample 'num_neg' negative items
      random_item = np.random.choice(unqiue_items)
      while (user, random_item) in interactions:
        random_item = np.random.choice(unqiue_items)
      data_with_negatives.append({'user': user, 'item': random_item, 'label': 0})

    # 3. Return a dataframe with columns: user, item, label
    return pd.DataFrame(data_with_negatives)

## PyTorch Dataset

We now wrap our processed data into a PyTorch `Dataset`.

This allows us to:
- Access individual samples as `(user, item, label)`
- Easily plug into a `DataLoader` for batching

You do not need to modify this part.

In [7]:
class InteractionDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['user'].values)
        self.items = torch.tensor(df['item'].values)
        self.labels = torch.tensor(df['label'].values).float()

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]

## DataLoader

The `DataLoader` will:
- Batch the data
- Shuffle the training data
- Feed it to the model during training

In [8]:
train_data = sample_negatives(train_df)

train_loader = DataLoader(InteractionDataset(train_data), batch_size=256, shuffle=True)

Why are we sampling negatives only for the training data?

## Quick Exploration

Before building models, take a moment to explore the data.

Try to understand:
- How many interactions each user has
- How popular certain items are

This can give intuition about the dataset.

In [9]:
# Interactions per user
user_counts = df['user_id'].value_counts()
print(user_counts.describe())

# Interactions per item
item_counts = df['item_id'].value_counts()
print(item_counts.describe())

count    942.000000
mean      58.784501
std       54.696664
min        3.000000
25%       19.000000
50%       39.500000
75%       80.750000
max      378.000000
Name: count, dtype: float64
count    1447.000000
mean       38.268832
std        57.956847
min         1.000000
25%         3.000000
50%        13.000000
75%        47.500000
max       501.000000
Name: count, dtype: float64


## Baseline Model: Matrix Factorization (MF)

We represent:
- Each **user** as a vector (embedding)
- Each **item** as a vector (embedding)

To predict interaction:
- Compute the **dot product** between user and item embeddings

This gives a **score** indicating how likely the user is to interact with the item.

---

### Your Task

Implement a model that:
1. Learns user and item embeddings
2. Computes their dot product as the output score

In [10]:
class MF(nn.Module):
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()

        self.num_users = num_users
        self.num_items = num_items
        self.emb_dim = emb_dim

        # TODO:
        # - Define user embedding layer
        self.user_embed = nn.Embedding(num_users, emb_dim)
        # - Define item embedding layer
        self.item_embed = nn.Embedding(num_items, emb_dim)

    def forward(self, user, item):
        # TODO:
        # - Get user and item embeddings
        user_vec = self.user_embed(user)
        item_vec = self.item_embed(item)
        # - Compute dot product
        score = (user_vec*item_vec).sum(dim=1)
        # - Return a single score per pair
        return score

## Training the MF Model

Now train your Matrix Factorization model.

You will need to:
- Define a loss function
- Define an optimizer
- Iterate over the DataLoader
- Update model parameters

---

### Hints

- Use **Binary Cross Entropy loss**
- Apply **sigmoid** to model outputs if needed
- Typical steps:
  - forward pass
  - compute loss
  - backward pass
  - optimizer step

In [11]:
def train(model, dataloader, epochs, lr):
    # TODO:
    # - Move model to device (optional)
    # - Define optimizer
    optimizer = torch.optim.SGD(model.parameters(), lr)
    # - Define loss function
    BCELoss = nn.BCELoss()

    losses = []  # track loss per epoch

    for epoch in range(epochs):
        total_loss = 0

        for user, item, label in dataloader:
            # TODO:
            # - Forward pass
            score = torch.sigmoid(model.forward(user, item)).squeeze()
            # - Compute loss
            loss = BCELoss(score, label)
            total_loss += loss.item()
            # - Backpropagation
            loss.backward()
            # - Optimizer step
            optimizer.step()

        # Print average loss per epoch

        losses.append(total_loss)
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

    return losses

## Evaluation (Hit@K)

We evaluate the model using a ranking-based metric.

For each user:
- Take one positive item
- Sample multiple negative items
- Rank all items using the model
- Check if the positive item is in the top-K

This is called **Hit@K**.

In [12]:
def hit_at_k(model, test_df, full_df, K=10, num_neg=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    model.to(device)

    hits = 0
    total = len(test_df) # We evaluate every positive interaction in the test set

    # Map interactions for quick lookup
    interacted_items = full_df.groupby('user_id')['item_id'].apply(set).to_dict()
    all_items = full_df['item_id'].unique()

    with torch.no_grad(): # Disable gradient calculation for speed/memory
        for _, row in test_df.iterrows():
            u = int(row['user_id'])
            pos_item = int(row['item_id'])

            # 1. Sample Negatives
            negatives = []
            while len(negatives) < num_neg:
                neg_item = np.random.choice(all_items)
                if neg_item not in interacted_items.get(u, set()):
                    negatives.append(neg_item)

            # 2. Prepare Tensors
            # We need a list of the 1 positive + 100 negatives
            item_list = [pos_item] + negatives
            user_tensor = torch.tensor([u] * (num_neg + 1)).to(device)
            item_tensor = torch.tensor(item_list).to(device)

            # 3. Get Scores
            scores = model(user_tensor, item_tensor).squeeze()

            # 4. Rank and Check Hit
            # We want to see if the item at index 0 (the positive) is in the top K
            # torch.topk returns values and indices of the highest scores
            _, top_indices = torch.topk(scores, K)

            top_indices = top_indices.cpu().numpy()
            if 0 in top_indices:
                hits += 1

    return hits / total

# TODO:
# 1. Initialize MF model
EPOCHS = 10
DIMS = 10
LEARNING_RATE = 0.007
model_MF = MF(len(df['user_id'].unique()), len(df['item_id'].unique()), DIMS)
# 2. Train it
train(model_MF, train_loader, EPOCHS, LEARNING_RATE)
# 3. Evaluate it
hit_at_k_score = hit_at_k(model_MF, test_df, df)
print(f"\nHit@K score for the MF model (epochs = {EPOCHS}, embedding dimesnsions = {DIMS}) =")
print(hit_at_k_score)

Epoch 1, Loss: 477.1523
Epoch 2, Loss: 377.9714
Epoch 3, Loss: 283.9023
Epoch 4, Loss: 261.0402
Epoch 5, Loss: 278.4303
Epoch 6, Loss: 299.6828
Epoch 7, Loss: 300.1193
Epoch 8, Loss: 277.7108
Epoch 9, Loss: 263.4675
Epoch 10, Loss: 280.9711

Hit@K score for the MF model (epochs = 10, embedding dimesnsions = 10) =
0.4714221218961625


If your score is low, try playing with the hyperparameters before moving on! Try sampling more negatives, or playing with the embedding dimensions.

Conversely, if your score is high, play with the values of K or number of negatives in evaluation (dec K, inc negatives).

## Neural Model: MLP-Based Recommender

Instead of using a dot product, we can learn a more flexible interaction function using a neural network.

Approach:
- Learn user and item embeddings
- **Concatenate** them
- Pass through a **Multi-Layer Perceptron (MLP)**

This allows the model to capture more complex relationships than simple similarity.

---

### Your Task

Implement a model that:
1. Learns user and item embeddings
2. Concatenates them
3. Passes them through an MLP to produce a score

The choice of activation functions is upto you.

In [13]:
class MLP(nn.Module):
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()

        self.num_users = num_users
        self.num_items = num_items
        self.emb_dim = emb_dim

        # TODO:
        # - Define user embedding layer
        self.user_embed = nn.Embedding(num_users, emb_dim)
        # - Define item embedding layer
        self.item_embed = nn.Embedding(num_items, emb_dim)
        # - Define MLP layers
        self.layers = nn.Sequential(
            nn.Linear(emb_dim*2, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, user, item):
        # TODO:
        # - Get embeddings
        user_vec = self.user_embed(user)
        item_vec = self.item_embed(item)
        # - Concatenate user and item embeddings
        concat_vec = torch.cat([user_vec, item_vec], dim=1)
        # - Pass through MLP
        score = self.layers(concat_vec)
        # - Return a single score
        return score

## Train and Evaluate the MLP Model

Repeat the same steps as before:
- Train the model
- Evaluate on the test set

Compare its performance with the MF model.

In [39]:
# TODO:
# 1. Initialize MLP model
EPOCHS = 10
DIMS = 20
LEARNING_RATE = 0.0032
model_MLP = MLP(len(df['user_id'].unique()), len(df['item_id'].unique()), DIMS)
# 2. Train it
train(model_MLP, train_loader, EPOCHS, LEARNING_RATE)
# 3. Evaluate it
hit_at_k_score = hit_at_k(model_MLP, test_df, df)
print(f"\nHit@K score for the MLP model (epochs = {EPOCHS}, embedding dimesnsions = {DIMS}) =")
print(hit_at_k_score)

Epoch 1, Loss: 240.3505
Epoch 2, Loss: 237.7271
Epoch 3, Loss: 230.6644
Epoch 4, Loss: 218.7733
Epoch 5, Loss: 205.1619
Epoch 6, Loss: 197.4734
Epoch 7, Loss: 192.3011
Epoch 8, Loss: 193.8530
Epoch 9, Loss: 198.4479
Epoch 10, Loss: 201.5118

Hit@K score for the MLP model (epochs = 10, embedding dimesnsions = 20) =
0.436568848758465


## Comparison & Analysis

You have now implemented:
- Matrix Factorization (MF)
- MLP-based recommender

---

### Compare the Models

- What score did each model achieve?
- Which model performed better?

---

### Think & Reflect

- Why might the MLP model outperform MF?
- In what cases might MF perform just as well or better?
- How does embedding size affect performance?
- Did you observe any signs of overfitting?

---

### Some Experiments

If you want to explore further:
- Try different embedding dimensions
- Change number of MLP layers
- Try different activation functions

---

## Bonus Task: Neural Collaborative Filtering (NCF)

In this task, you implemented:
- Matrix Factorization (MF)
- MLP-based recommender

The paper [Neural Collaborative Filtering](https://arxiv.org/pdf/1708.05031) proposes combining both ideas.

---

### Idea

- MF captures **linear interactions**
- MLP captures **nonlinear interactions**

NCF combines both by:
1. Computing MF output
2. Computing MLP output
3. Combining them into a final prediction

---

### Your Task

Design a model that:
- Uses both MF and MLP components
- Combines their outputs
- Trains end-to-end

---

### Hints

- You can:
  - Concatenate MF and MLP representations
  - Or combine their final scores
- Think about:
  - Should embeddings be shared or separate?
  - How to balance both components?

---

Does combining both approaches improve performance over MF and MLP individually?

In [40]:
"""
Bonus Tak: Neural Collaborative Filtering (NCF)

Implementing the NeuMF model as explained in the NCF paper, using the learnt parameters from
the pre-trained MF and MLP models above as the initializations for the paramters of the corresponding
MF and MLP components of the NeuMF model.

Moreover, as mentioned in the paper, the weights belonging to the final hidden layers for each of the corresponding components
has been concatenated into a single weight vector and the corresponding values have also been concatenated.
The trade-off between the weight vector of the MF and MLP models, respectively, in the concatenated weight vector is
determined using the hyperparamter alpha. We then apply the sigmoid function after applying the concatenated weight vector
on the concatenated value matrix to obtain the final score.
"""

'\nBonus Tak: Neural Collaborative Filtering (NCF)\n\nImplementing the NeuMF model as explained in the NCF paper, using the learnt parameters from\nthe pre-trained MF and MLP models above as the initializations for the paramters of the corresponding\nMF and MLP components of the NeuMF model.\n\nMoreover, as mentioned in the paper, the weights belonging to the final hidden layers for each of the corresponding components\nhas been concatenated into a single weight vector and the corresponding values have also been concatenated.\nThe trade-off between the weight vector of the MF and MLP models, respectively, in the concatenated weight vector is\ndetermined using the hyperparamter alpha. We then apply the sigmoid function after applying the concatenated weight vector\non the concatenated value matrix to obtain the final score.\n'

In [41]:
# Implementing the NeuMF class

import copy

class NeuMF(nn.Module):
  def __init__(self, alpha, pre_trained_MF, pre_trained_MLP):
    super().__init__()

    self.alpha = alpha

    # Initializing the user and item embedding layers for the individual MF and MLP components
    # using the learnt embeddings from the pre-trained models
    self.user_embed_MF = nn.Embedding(pre_trained_MF.num_users, pre_trained_MF.emb_dim)
    self.item_embed_MF = nn.Embedding(pre_trained_MF.num_items, pre_trained_MF.emb_dim)

    self.user_embed_MLP = nn.Embedding(pre_trained_MLP.num_users, pre_trained_MLP.emb_dim)
    self.item_embed_MLP = nn.Embedding(pre_trained_MLP.num_items, pre_trained_MLP.emb_dim)

    self.user_embed_MF.load_state_dict(pre_trained_MF.user_embed.state_dict())
    self.user_embed_MLP.load_state_dict(pre_trained_MLP.user_embed.state_dict())

    self.item_embed_MF.load_state_dict(pre_trained_MF.item_embed.state_dict())
    self.item_embed_MLP.load_state_dict(pre_trained_MLP.item_embed.state_dict())

    # Initializing the paramters for the hidden layers of the MLP component using the
    # corresponding parameters from the pre-trained MLP model (The list of layers is sliced
    # in order to exculde copying the last layer of the MLP model)
    self.component_MLP_layers = copy.deepcopy(nn.Sequential(*list(pre_trained_MLP.layers.children())[:-1]))

    # Defining a linear layer for the final score of the NeuMF model, obtained after applying
    # the concatenated weight vector on the concatenated values vector from the last hidden layers
    self.score_layer = nn.Linear((pre_trained_MF.emb_dim + list(self.component_MLP_layers.children())[-2].out_features), 1)

    # Concatenating the weight vectors for the last hidden layers
    with torch.no_grad():
      # Using the notation from the NCF paper

      # The weight vector for the embeddings in the MF component is simply a vector of ones
      h_mf = torch.ones(1, pre_trained_MF.emb_dim)
      # Whereas that for the last hidden layer of the MLP component is as follows:
      h_mlp = list(pre_trained_MLP.layers.children())[-1].weight
      # Concatenating the two with the trade-off determined by alpha
      h = torch.cat([self.alpha*h_mf, (1 - self.alpha)*h_mlp], dim=1)
      # Setting this as the weight vector for the score layer
      self.score_layer.weight.copy_(h)

  def forward(self, user, item):
    # Getting the embeddings for the user and item separately for the MF and MLP components
    user_vec_MF = self.user_embed_MF(user)
    item_vec_MF = self.item_embed_MF(item)

    user_vec_MLP = self.user_embed_MLP(user)
    item_vec_MLP = self.item_embed_MLP(item)

    # MF component using the element-wise product
    element_wise_MF = user_vec_MF*item_vec_MF
    # MLP component using all but the last layer from the pre-trained MLP model
    pre_output_MLP = self.component_MLP_layers(torch.cat([user_vec_MLP, item_vec_MLP], dim=1))
    # Concatenating the two
    pre_output_concat = torch.cat([element_wise_MF, pre_output_MLP], dim=1)
    # Evaluating using the concatenated weight vector h
    score = self.score_layer(pre_output_concat)

    return score

In [43]:
# Creating an instance of the NeuMF model and training it

# Initializing the NeuMF model
EPOCHS = 10
LEARNING_RATE = 0.004
ALPHA = 0.5
model_NeuMF = NeuMF(ALPHA, model_MF, model_MLP)
# Training it
train(model_NeuMF, train_loader, EPOCHS, LEARNING_RATE)
# Evaluating it
hit_at_k_score = hit_at_k(model_NeuMF, test_df, df)
print(f"\nHit@K score for the NeuMF (NCF) model (epochs = {EPOCHS}, alpha = {ALPHA}) =")
print(hit_at_k_score)

Epoch 1, Loss: 183.5309
Epoch 2, Loss: 178.5361
Epoch 3, Loss: 176.4476
Epoch 4, Loss: 175.3223
Epoch 5, Loss: 174.0803
Epoch 6, Loss: 169.7983
Epoch 7, Loss: 164.1036
Epoch 8, Loss: 157.4241
Epoch 9, Loss: 152.4580
Epoch 10, Loss: 150.0406

Hit@K score for the NeuMF (NCF) model (epochs = 10, alpha = 0.5) =
0.5157562076749436
